In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyreadr
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

In [3]:
claims_raw = pyreadr.read_r("../data/claims-raw.RData")['claims_raw']
claims_test = pyreadr.read_r("../data/claims-test.RData")['claims_test']

In [5]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])  

In [6]:
def parse_html(series, keep_headers = False):
    if keep_headers:
        tags = ["p", "h1", "h2", "h3", "h4", "h5", "h6"]
    else:
        tags = ["p"]
    
    def extract(html):
        soup = BeautifulSoup(html, "html.parser")
        txt = " ".join(tag.get_text(" ", strip=True) 
                       for tag in soup.find_all(tags))
        return txt
    
    return (
        series
        .apply(extract)
        .str.replace(r"http\S+|www\S+", " ", regex=True)
        .str.replace(r"\S+@\S+", " ", regex=True)
        .str.replace("'", "", regex=False)
        .str.replace(r"[\n]|[^\w\s]|nbsp|\d|[^\w\s]", " ", regex=True)
        .str.lower()
        .str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


In [7]:
claims_clean = claims_raw[claims_raw["text_tmp"].str.contains("<!", na=False)].copy()
claims_clean["text_clean"] = parse_html(claims_clean["text_tmp"])

In [8]:
claims_clean.to_pickle("../data/claims-clean.pkl", compression = "gzip")